# Moment Matrices and the Christoffel-Darboux Polynomial

This notebook explores the concept of moment matrices constructed from a small dataset of points. We will compare three different methods for constructing this matrix:

1.  **Empirical Measure**: The standard approach where the matrix is a sum of outer products of the feature vectors of the data points. This matrix is often rank-deficient.
2.  **Ball Measure (Smoothed)**: A "smoothed" version where we integrate the monomial features over a small ball around each data point. 
3.  **Gaussian Measure (Smoothed)**: Another smoothed version where we take the expectation of the monomial features with respect to a Gaussian distribution centered at each data point.

We will then compute and visualize the Christoffel-Darboux (CD) polynomial for each case. The CD polynomial, defined as $K(x,x) = \phi(x)^T M^{-1} \phi(x)$, provides a measure of leverage of point $x$. Its level sets can be interpreted as a data-driven geometry.

**Key Observation**: We expect the empirical matrix to be singular (rank-deficient), while the smoothed versions should be full-rank, leading to well-defined CD polynomials without explicit regularization.

## 1. Setup and Helper Functions

In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import itertools
from scipy.special import gamma as gamma_func, factorial2, comb
from scipy import integrate
import matplotlib.pyplot as plt
import seaborn as sns

N_DIM = 1      # n: Dimensionality of the space (1 or 2)
M_POINTS = 3   # m: Number of data points
DEGREE = 3     # d: Maximum degree of monomials

DATA_RADIUS = 1.0 # Radius of ball to sample points from
EPSILON = 0.5     # epsilon: Radius for the ball measure
SIGMA = 0.25
LAMBDA_REG = 1e-6 # Regularization for the empirical matrix

In [ ]:
from algs.bases.poly_basis import total_degree_index_set as get_monomial_basis

def phi(x, basis):
    """Computes the monomial feature vector for a point x."""
    x = np.asarray(x)
    return np.array([np.prod(x**alpha) for alpha in basis])

def sample_points_from_ball(m, n, r):
    """Samples m points uniformly from an n-dimensional ball of radius r."""
    # Sample from a multivariate normal, then normalize and scale
    points = np.random.randn(m, n)
    norms = np.linalg.norm(points, axis=1, keepdims=True)
    unit_vectors = points / norms
    
    # Scale by a radius sampled to ensure uniformity
    radii = r * (np.random.rand(m, 1)**(1/n))
    return unit_vectors * radii

## 2. Data Generation

We generate $m=3$ points from a ball of radius $r=1$ in $n=2$ dimensions.

In [ ]:
np.random.seed(2)

plt.figure(figsize=(6, 6))
if N_DIM == 2:
    points = sample_points_from_ball(M_POINTS, N_DIM, DATA_RADIUS)

else:
    points = np.random.uniform(low=-EPSILON, high=EPSILON, size=(M_POINTS,1))

## 3. Moment Matrix Implementations

We now create the monomial basis and then construct the three different moment matrices. We will check the rank of each matrix.

In [ ]:
# Get the monomial basis for our chosen dimension and degree
monomial_basis = get_monomial_basis(N_DIM, DEGREE)
basis_size = len(monomial_basis)
print(f"Dimension: {N_DIM}, Degree: {DEGREE}")
print(f"Monomial basis has {basis_size} elements:")
print(monomial_basis)

### 3.1. Empirical Moment Matrix

In [ ]:
M_emp = np.zeros((basis_size, basis_size))
for p in points:
    phi_p = phi(p, monomial_basis).reshape(-1, 1)
    M_emp += phi_p @ phi_p.T

rank_emp = np.linalg.matrix_rank(M_emp)
print(f"Empirical Moment Matrix Shape: {M_emp.shape}")
print(f"Rank of Empirical Matrix: {rank_emp} (Expected: <= {M_POINTS})")

We either use numerical integration (`scipy.integrate.nquad`) to compute them, or the analytic formula. 

### 3.2. Ball Measure Moment Matrix

Here, each entry $(\alpha, \beta)$ of the matrix is given by:
$$ [\tilde{M}_{d,\epsilon}]_{\alpha,\beta} = \sum_{i=1}^m \frac{1}{|B(x_i,\epsilon)|} \int_{B(x_i, \epsilon)} x^{\alpha+\beta} dx $$

In [ ]:
from algs.cd_smoothing import create_smooth_gauss, create_smooth_ball

M_gauss_analytic = create_smooth_gauss(points, DEGREE, SIGMA, analytic=True)
M_ball_analytic = create_smooth_ball(points, DEGREE, EPSILON, analytic=True)

print("shape and rank Gauss (analytic):", M_gauss_analytic.shape, np.linalg.matrix_rank(M_gauss_analytic))
print("shape and rank ball (analytic): ", M_ball_analytic.shape, np.linalg.matrix_rank(M_ball_analytic))

### 3.3. Gaussian Measure Moment Matrix

Here, each entry $(\alpha, \beta)$ is the sum of expectations:
$$ [\tilde{M}_{d,\sigma}]_{\alpha,\beta} = \sum_{i=1}^m E_{x \sim N(x_i, \sigma^2 I)}[x^{\alpha+\beta}] $$

In [ ]:
M_gauss_numeric = create_smooth_gauss(points, DEGREE, SIGMA, analytic=False)
M_ball_numeric = create_smooth_ball(points, DEGREE, EPSILON, analytic=False)

print("shape and rank Gauss (numeric):", M_gauss_numeric.shape, np.linalg.matrix_rank(M_gauss_numeric))
print("shape and rank ball (numeric):", M_ball_numeric.shape, np.linalg.matrix_rank(M_ball_numeric))

## 4. Christoffel-Darboux Polynomial Calculation and Visualization

The CD polynomial is $K(x,x) = \phi(x)^T M^{-1} \phi(x)$. We will calculate the inverse of each matrix (regularizing the empirical one) and then evaluate this expression over a grid to visualize the result.

In [ ]:
from scipy.linalg import cho_factor, cho_solve

def calculate_cd_polynomial(C, basis, coords, points):
    """Calculates the CD polynomial on a grid."""
    if len(basis[0])==2:
        cd_grid = []
        for point in zip(coords[0].flatten(), coords[1].flatten()):
            phi_x = phi(point, basis)
            cd_grid.append(phi_x.T @ cho_solve(C, phi_x))
        cd_grid = np.array(cd_grid).reshape(coords[0].shape)

        cd_points = []
        for point in points:
            phi_x = phi(point, basis)
            cd_points.append(phi_x.T @ cho_solve(C, phi_x))
        cd_points = np.array(cd_points)
    else:
        cd_grid = []
        for point in coords.flatten():
            phi_x = phi(point, basis)
            cd_grid.append(phi_x.T @ cho_solve(C, phi_x))
        cd_grid = np.array(cd_grid)
        
        
        cd_points = []
        for point in points:
            phi_x = phi(point, basis)
            cd_points.append(phi_x.T @ cho_solve(C, phi_x))
        cd_points = np.array(cd_points)
    return cd_grid, cd_points


def get_grid(points, padding=1.0):
    """Creates a 2D grid for plotting."""
    x_min, y_min = points.min(axis=0) - padding
    x_max, y_max = points.max(axis=0) + padding
    grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, 100),
                                 np.linspace(y_min, y_max, 100))
    return grid_x, grid_y

def plot_cd_polynomial(coords, cd_coords, points, cd_points, title):
    """Plots the CD polynomial as a contour plot."""
    if points.shape[1] == 2:
        plt.contourf(coords[0], coords[1], np.log10(cd_coords), levels=50, cmap='viridis')
        plt.colorbar(label='$K(x,x)$')
        plt.scatter(points[:, 0], points[:, 1], c='red', s=10, zorder=5)
        plt.xlabel('$x_1$')
        plt.ylabel('$x_2$')
        plt.axis('equal')
        plt.axis("off")
    else: 
        plt.semilogy(coords.flatten(), cd_coords.flatten())
        plt.scatter(points.flatten(), cd_points.flatten(), color='red')
        plt.xlabel("$x$")
    plt.title(title)

In [ ]:
padding = 1.5
if N_DIM == 2:
    coords = get_grid(points, padding=padding)
else:
    coords = np.linspace(np.min(points)-padding, np.max(points+padding), 100)

# Calculate CD polynomials for each case
lambda_reg = 1e-6
C_emp = cho_factor(M_emp + lambda_reg * np.eye(M_emp.shape[0]))
C_ball = cho_factor(M_ball_analytic)
C_gauss = cho_factor(M_gauss_analytic)

cd_emp_grid, cd_emp_points = calculate_cd_polynomial(C_emp, monomial_basis, coords, points)
cd_ball_grid, cd_ball_points = calculate_cd_polynomial(C_ball, monomial_basis, coords, points)
cd_gauss_grid, cd_gauss_points = calculate_cd_polynomial(C_gauss, monomial_basis, coords, points)

# Plotting
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
plt.sca(axes[0])
plot_cd_polynomial(coords, cd_emp_grid, points, cd_emp_points, f'Empirical')

plt.sca(axes[1])
plot_cd_polynomial(coords, cd_ball_grid, points, cd_ball_points, f'Ball analytic')

plt.sca(axes[2])
plot_cd_polynomial(coords, cd_gauss_grid, points, cd_gauss_points, f'Gaussian analytic')

plt.tight_layout()
plt.show()

## 5. Conclusion

As expected, the results demonstrate the following:

- **Rank**: The empirical moment matrix is rank-deficient (rank 3, equal to the number of points), as it is constructed from only 3 independent vectors in a 10-dimensional feature space. In contrast, both the ball and Gaussian measure matrices are full-rank (rank 10). This is because the integration/expectation step introduces new information and breaks the linear dependencies.

- **Christoffel-Darboux Polynomial**: 
    - The CD polynomial for the **empirical case** is heavily influenced by the regularization parameter and its shape is highly concentrated around the data points, quickly decaying to zero elsewhere. The geometry it defines is somewhat degenerate.
    - The CD polynomials for the **smoothed measures** are well-behaved and provide a much more stable and smooth geometry. The shape of their level sets envelops all the data points, defining a region of interest based on the data. The `epsilon` parameter controls the "tightness" of this shape around the points.